# TTA Example

## Imports and Configs

In [ ]:
import os
os.chdir("/home/work/t2a_voicestudio/tta/tta")
print(os.getcwd())

In [ ]:
import sys
from os import path, environ
from argparse import ArgumentParser

import torch
from torchinfo import summary

from ttadapters import datasets, models, methods
from ttadapters.utils import visualizer, validator
from ttadapters.datasets import DatasetHolder, scenarios

from torch.utils.data import DataLoader

In [ ]:
environ["TORCHDYNAMO_CAPTURE_SCALAR_OUTPUTS"] = "1"
environ["TORCHDYNAMO_CAPTURE_DYNAMIC_OUTPUT_SHAPE_OPS"] = "1"

torch._dynamo.config.capture_scalar_outputs = True
torch._dynamo.config.suppress_errors = True

In [ ]:
environ["CUDA_VISIBLE_DEVICES"] = "0"

### Parse Arguments

In [ ]:
# Set Batch Size
BATCH_SIZE = 2, 8, 1  # Local
#BATCH_SIZE = 40, 200, 1  # A100 or H100
ACCUMULATE_STEPS = 1

# Set Data Root
DATA_ROOT = path.join(".", "data")

# Set Target Dataset
SOURCE_DOMAIN = datasets.SHIFTDataset

# Set Model List
MODEL_ZOO = ["rcnn", "swinrcnn", "yolo11", "rtdetr"]
MODEL_TYPE = MODEL_ZOO[1]

# Set method
BASELINE = ["ActMAD", "NORM", "DUA", "TeST", "WHW"]
BASELINE_TYPE = BASELINE[3]
LR = 1e-9 # {rcnn, swinrcnn, yolo11, rtdetr | actmad : le-9}, {rcnn, swinrcnn, rtdetr | mean-teacher : 1e-9}

In [ ]:
# Create argument parser
parser = ArgumentParser(description="Adaptation experiment script for Test-Time Adapters")

# Add model arguments
parser.add_argument("--dataset", type=str, choices=["shift", "city"], default="shift", help="Training dataset")
parser.add_argument("--model", type=str, choices=MODEL_ZOO, default=MODEL_TYPE, help="Model architecture")

# Add training arguments
parser.add_argument("--train-batch", type=int, default=BATCH_SIZE[0], help="Training batch size")
parser.add_argument("--valid-batch", type=int, default=BATCH_SIZE[1], help="Validation batch size")
parser.add_argument("--accum-step", type=int, default=ACCUMULATE_STEPS, help="Gradient accumulation steps")
parser.add_argument("--data-root", type=str, default=DATA_ROOT, help="Root directory for datasets")
parser.add_argument("--device", type=int, default=0, help="CUDA device number")
parser.add_argument("--additional_gpu", type=int, default=0, help="Additional CUDA device count")
parser.add_argument("--use-bf16", action="store_true", help="Use bfloat16 precision")

# Parsing arguments
if "ipykernel" in sys.modules:
    args = parser.parse_args([])
    print("INFO: Running in notebook mode with default arguments")
else:
    args = parser.parse_args()

# Update global variables based on parsed arguments
BATCH_SIZE = args.train_batch, args.valid_batch, BATCH_SIZE[2]
ACCUMULATE_STEPS = args.accum_step
DATA_ROOT = args.data_root
MODEL_TYPE = args.model
match args.dataset:
    case "shift":
        SOURCE_DOMAIN = datasets.SHIFTDataset
    case "city":
        SOURCE_DOMAIN = datasets.CityScapesDataset
    case _:
        raise ValueError(f"Unsupported dataset: {args.dataset}")
print(f"INFO: Set batch size - Train: {BATCH_SIZE[0]}, Valid: {BATCH_SIZE[1]}, Test: {BATCH_SIZE[2]}")

### Check GPU Availability

In [ ]:
!nvidia-smi

In [ ]:
# Set CUDA Device Number
DEVICE_NUM = 0 if not args.device else args.device
ADDITIONAL_GPU = 0 if not args.additional_gpu else args.additional_gpu
DATA_TYPE = torch.float32 if not args.use_bf16 else torch.bfloat16

if torch.cuda.is_available():
    if ADDITIONAL_GPU:
        torch.cuda.set_device(DEVICE_NUM)
        device = torch.device("cuda")
    else:
        device = torch.device(f"cuda:{DEVICE_NUM}")
else:
    device = torch.device("cpu")
    DEVICE_NUM = -1

print(f"INFO: Using device - {device}" + (f":{DEVICE_NUM}" if ADDITIONAL_GPU else ""))
print(f"INFO: Using data precision - {DATA_TYPE}")

## Define Dataset

In [ ]:
# Fast download patch
datasets.patch_fast_download_for_object_detection()

In [ ]:
# Dataset info
CLASSES = ['pedestrian', 'car', 'truck', 'bus', 'motorcycle', 'bicycle']
NUM_CLASSES = len(CLASSES)
print(f"INFO: Number of classes - {NUM_CLASSES} {CLASSES}")

## Load Base Model

In [ ]:
# Initialize base_model
# USE_ADAPTER: True면 생성 시 adapter 부착, False면 기존 방식
USE_ADAPTER = True
ADAPTER_RATIO = 32  # adapter bottleneck ratio

match MODEL_TYPE:
    case "rcnn":
        base_model = models.FasterRCNNForObjectDetection(dataset=SOURCE_DOMAIN)
        
        load_result = base_model.load_from(
            **vars(base_model.Weights.SHIFT_CLEAR_NATUREYOO if SOURCE_DOMAIN == datasets.SHIFTDataset else base_model.Weights.CITYSCAPES), 
            strict=False
        )
    case "swinrcnn":
        base_model = models.SwinRCNNForObjectDetection(dataset=SOURCE_DOMAIN)
        load_result = base_model.load_from(
            **vars(base_model.Weights.SHIFT_CLEAR_NATUREYOO if SOURCE_DOMAIN == datasets.SHIFTDataset else base_model.Weights.CITYSCAPES), 
            strict=False
        )
    case "yolo11":
        base_model = models.YOLO11ForObjectDetection(dataset=SOURCE_DOMAIN)
        load_result = base_model.load_from(**vars(base_model.Weights.SHIFT_CLEAR if SOURCE_DOMAIN == datasets.SHIFTDataset else base_model.Weights.CITYSCAPES), strict=False)
    case "rtdetr":
        DATA_TYPE = torch.bfloat16  # bf16 default
        base_model = models.RTDetrForObjectDetection(dataset=SOURCE_DOMAIN)
        load_result = base_model.load_from(**vars(base_model.Weights.SHIFT_CLEAR if SOURCE_DOMAIN == datasets.SHIFTDataset else base_model.Weights.CITYSCAPES), strict=False)
    case _:
        raise ValueError(f"Unsupported model type: {MODEL_TYPE}")
    
print(f"Model: {base_model.model_name}")
print(f"Load result: {load_result}")

In [ ]:
base_model.to(device)

In [ ]:
summary(base_model)

### Load Scenarios

In [ ]:
data_preparation = base_model.DataPreparation(datasets.base.BaseDataset(), evaluation_mode=True)

In [ ]:
from ttadapters.datasets.scenarios.continual import DiscreteSubsetType

CUSTOM_ORDER = [
    DiscreteSubsetType.OVERCAST_DAYTIME,
    DiscreteSubsetType.RAINY_DAYTIME,
    DiscreteSubsetType.CLEAR_NIGHT,
    DiscreteSubsetType.FOGGY_DAYTIME,
]

In [ ]:
continual_scenario = scenarios.SHIFTDiscreteScenarioForContinualTTA(
    root=DATA_ROOT, valid=True, transforms=data_preparation.transforms,
    order=scenarios.SHIFTDiscreteScenarioForContinualTTA.WHWPAPER
)

## BASELINE

In [ ]:
from ttadapters.methods.baseline.method import ActMADConfig, ActMADEngine, NORMEngine, NORMConfig, DUAEngine, DUAConfig, TeSTEngine, TeSTConfig, WHWConfig, WHWEngine

if BASELINE_TYPE == "ActMAD":
    if MODEL_TYPE == "rcnn":
        config = ActMADConfig(
            model_type=MODEL_TYPE,
            adaptation_layers="backbone",
            optim="SGD",
            adapt_lr=LR,
            loss_type="L1",
            statistic_save_path=f"/home/work/t2a_voicestudio/tta/tta/ttadapters/methods/baseline/statistics/actmad_{MODEL_TYPE}_source_stats.pt",
        )
    elif MODEL_TYPE == "swinrcnn":
        config = ActMADConfig(
            model_type=MODEL_TYPE,
            adaptation_layers="backbone",
            optim="SGD",
            adapt_lr=LR,
            loss_type="L1",
            statistic_save_path=f"/home/work/t2a_voicestudio/tta/tta/ttadapters/methods/baseline/statistics/actmad_{MODEL_TYPE}_source_stats.pt",
        )
    elif MODEL_TYPE == "yolo11":
        config = ActMADConfig(
            model_type=MODEL_TYPE,
            adaptation_layers="backbone",
            optim="SGD",
            adapt_lr=LR,
            loss_type="L1",
            statistic_save_path=f"/home/work/t2a_voicestudio/tta/tta/ttadapters/methods/baseline/statistics/actmad_{MODEL_TYPE}_source_stats.pt",
        )
    elif MODEL_TYPE == "rtdetr":
        config = ActMADConfig(
            model_type=MODEL_TYPE,
            adaptation_layers="backbone+encoder",
            optim="SGD",
            adapt_lr=LR,
            loss_type="L1",
            statistic_save_path=f"/home/work/t2a_voicestudio/tta/tta/ttadapters/methods/baseline/statistics/actmad_{MODEL_TYPE}_source_stats.pt",
        )        
    adaptive_model = ActMADEngine(base_model, config)
    source_dataset = datasets.SHIFTClearDatasetForObjectDetection(root=DATA_ROOT, train=True)
    source_prep = base_model.DataPreparation(source_dataset, evaluation_mode=True)
    adaptive_model.fit(source_prep, batch_size=4)
    
elif BASELINE_TYPE == "NORM":
    if MODEL_TYPE in ["rcnn", "yolo11"]:
        config = NORMConfig(model_type=MODEL_TYPE, source_sum=128, adaptation_layers="backbone")
    elif MODEL_TYPE == "rtdetr":
        config = NORMConfig(model_type=MODEL_TYPE, source_sum=128, adaptation_layers="backbone+encoder")
    else:
        raise NotImplementedError("This model cannot be applied to this method.") 
    adaptive_model = NORMEngine(base_model, config)

elif BASELINE_TYPE == "DUA":
    if MODEL_TYPE in ["rcnn", "yolo11"]:
        config = DUAConfig(model_type=MODEL_TYPE, adaptation_layers="backbone")
    elif MODEL_TYPE == "rtdetr":
        config = DUAConfig(model_type=MODEL_TYPE, adaptation_layers="backbone+encoder")
    else:
        raise NotImplementedError("This model cannot be applied to this method.") 
    adaptive_model = DUAEngine(base_model, config)

elif BASELINE_TYPE == "TeST":
    config = TeSTConfig(                                                                                                                                                                            
        model_type=MODEL_TYPE,
        optim="SGD",
        adapt_lr=LR,
        stage="online",
        lambda_cons=1.0, # feature consistency loss weight
        lambda_ent=0.5, # entropy loss weight
        n_teacher_epochs=10, # Stage 1 epoch
        n_student_steps=10 # Stage 2 epoch
    )
    adaptive_model = TeSTEngine(base_model, config)

elif BASELINE_TYPE == "WHW":
    if MODEL_TYPE in ["rcnn", "swinrcnn"]:
        whw_config = WHWConfig(
            source_stats_path=f"/home/work/t2a_voicestudio/tta/tta/ttadapters/methods/baseline/statistics/whw_{MODEL_TYPE}_source_stats.pt",
            backbone=MODEL_TYPE,
            skip_redundant="stat-period-ema",
            freq_weight=True,
            adapt_lr=LR
        )
    else :
        raise NotImplementedError("This model cannot be applied to this method.")
    adaptive_model = WHWEngine(base_model, whw_config)
    source_dataset = datasets.SHIFTClearDatasetForObjectDetection(root=DATA_ROOT, train=True)
    source_prep = base_model.DataPreparation(source_dataset, evaluation_mode=True)
    adaptive_model.fit(source_prep, batch_size=4)

### Setup WHW TTA

In [ ]:
adaptive_model.to(device)
adaptive_model.online()
summary(adaptive_model)

### Evaluate with Baseline

In [ ]:
tta = methods.MethodContainer(**{
    BASELINE_TYPE : adaptive_model
})

evaluator = validator.DetectionEvaluator(tta.methods(), classes=CLASSES, data_preparation=data_preparation, dtype=DATA_TYPE, device=device, no_grad=False)
evaluator_loader_params = dict(batch_size=BATCH_SIZE[2], shuffle=False, collate_fn=data_preparation.collate_fn)

In [ ]:
TOTAL_ROUNDS = 10

adaptation_results = []

for this in range(TOTAL_ROUNDS):
    result = visualizer.visualize_metrics(continual_scenario(**evaluator_loader_params).play(evaluator, index=tta.names()))
    adaptation_results.append(result)